# Code to reproduce predictions obtained from this model.

Note: I have not run through these cells entirely as I dont have the datasets stored locally or the compute to actually do the training. If you (the examiner/moderator) have any issue with running this that I haven't foreseen, please feel free to contact me. Google Colab notebooks in notebooks/training also run through this, but in a less ordered/formatted way.

In [ ]:
import sys
from pathlib import Path
import torch
CODE_DIR = Path("../../code").resolve()
sys.path.insert(0, str(CODE_DIR))

# Mask R-CNN

### Dataset
This [dataset](https://drive.google.com/drive/folders/1w9F2HbTzQ7DbAh4oUd7-YH5wfBN3375j?usp=sharing) is already compiled. Download the parent directory (```Tiles```). The dataset was created by hand labelling large DEM scenes, and exporting training tiles using ScaleNormalisedDataStack.py. See code/notebooks/colab/DataPreperation for an example (or to run).

In [ ]:
dataset_root = '/path/to/Tiles/' # where Tiles is the parent directory and contains the folders such as ScalesCombined.

### Model

To train a new model, with a specific region folded out, follow the cells below.

In [ ]:
from modelling.MaskRCNN import MaskRCNN
from modelling.TrainingFunctions import train_for_deployment

model = MaskRCNN() # instantiates a new model
best_model = train_for_deployment(model, 'UK', dataset_root) # regions could be France, Russia, Karoo, Australia, Brazil, USA (used training regions)

model.load_state_dict(best_model)

To load a pretrained model instead, follow the cells below. A pretrained Mask R-CNN model is available [here](https://drive.google.com/file/d/1mLdQIOr7wFF5DMf49PcsMjyxwK-8L0-g/view?usp=sharing).

In [ ]:
from modelling.MaskRCNN import MaskRCNN

pretrained_state_path = 'path/to/weights/'
maskrcnn_state_dict = torch.load(pretrained_state_path)

# Veto Classifier

A pre-labelled training dataset can be found [here]

In [ ]:
veto_dataset_root = 'path/to/VetoTiles'

To train a new classifier:

In [ ]:
paths = glob.glob(f'{veto_dataset_root}/*/*/*.npz')
epochs = 50

from modelling.MaskRCN import VetoClassifier
from modelling.TrainingFunctions import train_veto_classifier

veto = VetoClassifier()
results, veto_best_state = train_veto_classifier(veto, epochs, paths)
veto.load_state_dict(best_state)

Alternatively, a pretrained Veto Classifier can be initialised. Weights can be found [here](https://drive.google.com/file/d/1dy4stfla93QKlDzqyNNMbU3MbfXmTWMr/view?usp=sharing).

In [ ]:
veto_best_state = torch.load('path/to/weights')

### Inference.

Example (correlating) DEM and RGB scenes from the Bristol Channel Fault Zone region are available [here](https://drive.google.com/drive/folders/12ZN8SewFaRC5t6njgYwXjDDFWkaSJTch?usp=sharing).

In [ ]:
# config, customisable

DEM_PATH = 'path/to/RGB/composite.tif'
RGB_PATH = 'path/to/DEM.tif'
RESOLUTIONS = [1, 3, 6, 9, 12, 15]  # metres per pixel sweeps
OUT_PATH = 'output/filename.shp'

deployer = Deployer(
        dem_path=temp_path,
        rgb_path=rgb_path,
        model_state_dict=mask_rcnn_state_dict,
        veto_model_dict=veto_best_state,
        device=device,
        resolutions=RESOLUTIONS,
        native_res = 1,
        veto_threshold=0.85)

# run inference
result = deployer.sweep(stride_frac=0.65)

# unpack outputs
(predictions, cleaned, rejected, veto_probability_map, out_transform, out_crs, support, coverage) = result

# merge predictions from different resolutions, and save shapefile.
deployer.merge_predictions(cleaned, out_transform, out_crs, output_path=OUT_PATH)